# Index Handling From Start to End

This notebook reconstructs the index layer by hand instead of importing `SPINOR_INDEX`, `LORENTZ_INDEX`, `COLOR_FUND_INDEX`, `COLOR_ADJ_INDEX`, `WEAK_FUND_INDEX`, `WEAK_ADJ_INDEX`, or `flavor_index(...)`.

It mirrors the actual code path in:
- `src/symbolic/spenso_structures.py`
- `src/model/metadata.py`
- `src/model/lowering.py`
- `src/model/interactions.py`
- `src/symbolic/vertex_engine.py`

The goal is to make the pipeline explicit:

1. raw Spenso `Representation` objects
2. project-level `IndexType` declarations
3. typed Spenso tensor builders (`gamma`, `t`, `f`, metrics)
4. `Field` declarations carrying tuples of `IndexType`
5. lowering into `InteractionTerm`
6. conversion to `vertex_factor(...)` inputs
7. final model-level Feynman rules


In [2]:
import sys
from pathlib import Path
from fractions import Fraction
from pprint import pprint

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from symbolica import Expression, S
from symbolica.community.spenso import (
    LibraryTensor,
    Representation,
    Slot,
    TensorLibrary,
    TensorName,
    TensorNetwork,
    TensorStructure,
)

from model import (
    CovD,
    Field,
    GaugeGroup,
    GaugeRepresentation,
    Gamma,
    IndexType,
    Model,
    Parameter,
)
from model.lowering import _lower_local_interaction_monomial
from symbolic.vertex_engine import I

print("Python:", sys.executable)
print("Repository root:", ROOT)


## 1. What comes from Spenso?

The codebase imports the following names from `symbolica.community.spenso`:

- `Representation`: builds typed slot families like bispinor, Minkowski/Lorentz, fundamental, and adjoint.
- `Slot`: the concrete typed slot object produced by a representation.
- `TensorName`: the typed tensor head used for `gamma`, `gamma5`, `t`, `f`, and custom tensors.
- `TensorLibrary`, `LibraryTensor`, `TensorStructure`, `TensorNetwork`: used when the project registers explicit invariant-tensor component data.

For index handling itself, the most important objects are `Representation`, `Slot`, and `TensorName`.


In [4]:
spenso_imports = {
    "Representation": "builds typed slot families such as bis(4), mink(4), cof(N), coad(N)",
    "Slot": "the concrete typed slot object produced by a representation",
    "TensorName": "the tensor head used for gamma, gamma5, t, f, and custom tensors",
    "TensorLibrary": "library of tensor components",
    "LibraryTensor": "component payload registered into a TensorLibrary",
    "TensorStructure": "slot signature used when registering explicit components",
    "TensorNetwork": "executor for tensor expressions against a library",
}

for name, purpose in spenso_imports.items():
    print(f"{name:14} -> {purpose}")


## 2. Rebuild the raw representation layer

This is the real bottom layer behind all project indices. A project `IndexType` is just metadata wrapped around one of these raw Spenso `Representation` objects.


In [6]:
SPINOR_KIND = "spinor"
LORENTZ_KIND = "lorentz"
COLOR_FUND_KIND = "color_fund"
COLOR_ADJ_KIND = "color_adj"
WEAK_FUND_KIND = "weak_fund"
WEAK_ADJ_KIND = "weak_adj"
GENERATION_KIND = "generation"

BISPINOR_REP = Representation.bis(4)
MINKOWSKI_REP = Representation.mink(4)
COLOR_FUND_REP = Representation.cof(3)
COLOR_ADJ_REP = Representation.coad(8)
WEAK_FUND_REP = Representation.cof(2)
WEAK_ADJ_REP = Representation.coad(3)
GENERATION_REP = Representation.cof(3)

raw_slots = {
    "spinor": BISPINOR_REP("i"),
    "lorentz": MINKOWSKI_REP("mu"),
    "color adjoint": COLOR_ADJ_REP("a"),
    "color fund": COLOR_FUND_REP("c"),
    "weak adjoint": WEAK_ADJ_REP("aw"),
    "weak fund": WEAK_FUND_REP("w"),
    "generation": GENERATION_REP("f"),
}

for label, slot in raw_slots.items():
    print(f"{label:14} -> {slot}")

print()
print("Representation class:", type(MINKOWSKI_REP).__name__)
print("Slot class:", type(raw_slots["lorentz"]).__name__)


## 3. Rebuild the project's `IndexType` objects

This is the class the model layer actually uses. Fields do **not** carry raw Spenso `Slot` objects. They carry tuples of `IndexType` declarations.


In [8]:
SPINOR_INDEX = IndexType("Spinor", BISPINOR_REP, SPINOR_KIND, prefix="i")
LORENTZ_INDEX = IndexType("Lorentz", MINKOWSKI_REP, LORENTZ_KIND, prefix="mu")
COLOR_FUND_INDEX = IndexType("ColorFund", COLOR_FUND_REP, COLOR_FUND_KIND, prefix="c")
COLOR_ADJ_INDEX = IndexType("ColorAdj", COLOR_ADJ_REP, COLOR_ADJ_KIND, prefix="a")
WEAK_FUND_INDEX = IndexType("WeakFund", WEAK_FUND_REP, WEAK_FUND_KIND, prefix="w")
WEAK_ADJ_INDEX = IndexType("WeakAdj", WEAK_ADJ_REP, WEAK_ADJ_KIND, prefix="aw")
GENERATION_INDEX = IndexType(
    "Generation",
    GENERATION_REP,
    GENERATION_KIND,
    dimension=3,
    is_flavor=True,
    prefix="f",
)

index_catalog = (
    ("Spinor", SPINOR_INDEX),
    ("Lorentz / Minkowski", LORENTZ_INDEX),
    ("Color fundamental", COLOR_FUND_INDEX),
    ("Color adjoint", COLOR_ADJ_INDEX),
    ("Weak fundamental", WEAK_FUND_INDEX),
    ("Weak adjoint", WEAK_ADJ_INDEX),
    ("Flavor / generation", GENERATION_INDEX),
)

for title, index in index_catalog:
    print(title)
    print("  class         :", type(index).__name__)
    print("  name          :", index.name)
    print("  kind          :", index.kind)
    print("  is_flavor     :", index.is_flavor)
    print("  prefix        :", index.prefix)
    print("  representation:", index.representation)
    print()

print("Important point: fields carry IndexType objects, not raw Slot objects.")


## 4. Typed Spenso slots and tensor builders

This mirrors the logic in `src/symbolic/spenso_structures.py`. The project takes plain labels like `mu`, `a`, `i`, `f` and upgrades them into typed Spenso slots only when it builds actual tensors.


In [10]:
def _slot(rep, index):
    if isinstance(index, Slot):
        return index
    return rep(index)


def bispinor_index(index):
    return _slot(BISPINOR_REP, index)


def lorentz_index(index):
    return _slot(MINKOWSKI_REP, index)


def color_fund_index(index):
    return _slot(COLOR_FUND_REP, index)


def color_adj_index(index):
    return _slot(COLOR_ADJ_REP, index)


def weak_fund_index(index):
    return _slot(WEAK_FUND_REP, index)


def weak_adj_index(index):
    return _slot(WEAK_ADJ_REP, index)


def gamma_matrix(left_spinor, right_spinor, lorentz):
    return TensorName.gamma()(
        bispinor_index(left_spinor),
        bispinor_index(right_spinor),
        lorentz_index(lorentz),
    ).to_expression()


def gamma5_matrix(left_spinor, right_spinor):
    return TensorName.gamma5()(
        bispinor_index(left_spinor),
        bispinor_index(right_spinor),
    ).to_expression()


def lorentz_metric(mu, nu):
    return MINKOWSKI_REP.g(mu, nu).to_expression()


def spinor_metric(i, j):
    return BISPINOR_REP.g(i, j).to_expression()


def color_generator(adj, fund_left, fund_right):
    return TensorName.t()(
        color_adj_index(adj),
        color_fund_index(fund_left),
        color_fund_index(fund_right),
    ).to_expression()


def color_structure_constant(a, b, c):
    return TensorName.f()(
        color_adj_index(a),
        color_adj_index(b),
        color_adj_index(c),
    ).to_expression()


def weak_generator(adj, fund_left, fund_right):
    return TensorName.t()(
        weak_adj_index(adj),
        weak_fund_index(fund_left),
        weak_fund_index(fund_right),
    ).to_expression()


def weak_structure_constant(a, b, c):
    return TensorName.f()(
        weak_adj_index(a),
        weak_adj_index(b),
        weak_adj_index(c),
    ).to_expression()


alpha, beta, mu, a, cbar, c = S("alpha", "beta", "mu", "a", "cbar", "c")

print("gamma(alpha, beta, mu) =", gamma_matrix(alpha, beta, mu))
print("g(mu, nu)              =", lorentz_metric(mu, S("nu")))
print("T(a, cbar, c)          =", color_generator(a, cbar, c))
print("f(a, b, c)             =", color_structure_constant(a, S("b"), S("c")))
print("weak T                 =", weak_generator(S("aw"), S("w1"), S("w2")))


## 5. Fields use tuples of `IndexType`, not raw Spenso slots

This is the main semantic layer used by the model package. A field declares **which kinds of indices it carries**, and only later do those kinds get concrete labels.


In [12]:
Gluon = Field(
    "G",
    spin=1,
    self_conjugate=True,
    symbol=S("G"),
    indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX),
)

WBoson = Field(
    "W",
    spin=1,
    self_conjugate=True,
    symbol=S("W"),
    indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX),
)

Higgs = Field(
    "H",
    spin=0,
    self_conjugate=False,
    symbol=S("H"),
    conjugate_symbol=S("Hdag"),
    indices=(WEAK_FUND_INDEX,),
)

QClass = Field(
    "Q",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("Q"),
    conjugate_symbol=S("Qbar"),
    indices=(GENERATION_INDEX, COLOR_FUND_INDEX, SPINOR_INDEX),
    flavor_index=GENERATION_INDEX,
    class_members=("u", "c", "t"),
)

LClass = Field(
    "L",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("L"),
    conjugate_symbol=S("Lbar"),
    indices=(GENERATION_INDEX, WEAK_FUND_INDEX, SPINOR_INDEX),
    flavor_index=GENERATION_INDEX,
    class_members=("e", "mu", "ta"),
)

RepeatedColorScalar = Field(
    "X",
    spin=0,
    self_conjugate=False,
    symbol=S("X"),
    conjugate_symbol=S("Xdag"),
    indices=(COLOR_FUND_INDEX, COLOR_FUND_INDEX),
)

for field in (Gluon, WBoson, Higgs, QClass, LClass, RepeatedColorScalar):
    print(f"{field.name:3} -> {field.indices}")

print()
print("Q generic indices     :", QClass.indices)
print("Q class-member indices:", [member.indices for member in QClass.class_members])


## 6. Labels are stored by kind, but attached by slot

Internally, the user-facing label format is kind-keyed (`{"spinor": alpha, ...}`), but the field knows how to convert that to and from concrete slot positions. Repeated kinds are kept in tuple order.


In [14]:
q_slot_labels = {
    0: S("f"),
    1: S("c"),
    2: S("alpha"),
}
packed_q = QClass.pack_slot_labels(q_slot_labels)
unpacked_q = QClass.unpack_slot_labels(packed_q)

repeated_color_labels = {
    0: S("c1"),
    1: S("c2"),
}
packed_repeated = RepeatedColorScalar.pack_slot_labels(repeated_color_labels)
unpacked_repeated = RepeatedColorScalar.unpack_slot_labels(packed_repeated)

print("Packed Q labels:")
pprint(packed_q)
print("Unpacked Q labels:")
pprint(unpacked_q)

print()
print("Packed repeated-color labels:")
pprint(packed_repeated)
print("Unpacked repeated-color labels:")
pprint(unpacked_repeated)


## 7. Flavor indices are ordinary `IndexType` objects with `is_flavor=True`

There is no separate flavor engine at the declaration layer. Flavor is just another index kind with a declared `dimension` and `is_flavor=True`. The `Field` class uses that metadata to build class members; flavor expansion uses it later at `feynman_rule(..., flavor_expand=True)`.

Single-index flavor parameters such as `y(f)` in `y(f) * l.bar(f) * l(f)` allow the same label to appear more than twice in one term by default (`Parameter.permits_label_summation()`). Set `allow_summation=False` to reject that pattern explicitly.


In [16]:
Yu = Parameter("Yu", indices=(GENERATION_INDEX, GENERATION_INDEX))
y_diag = Parameter("yDiag", indices=(GENERATION_INDEX,))

print("Parameter class:", type(Yu).__name__)
print("Yu(f, h)       :", Yu(S("f"), S("h")))
print("Q flavor index :", QClass.flavor_index)
print("Q flavor slot  :", QClass.flavor_index_slot())
print("Q members      :", [member.name for member in QClass.class_members])
print("yDiag permits label summation by default:", y_diag.permits_label_summation())

for member in QClass.class_members:
    print(f"{member.name:2} -> {member.indices}")


## 8. Local monomials are lowered by matching labels against `IndexType.kind`

This is the bridge from the declarative layer to `InteractionTerm`. We build a Yukawa-like local monomial with explicit generation, color, and spinor labels, then lower it.


In [18]:
f, h, c, alpha = S("f", "h", "c", "alpha")
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("Phi"))

yukawa_monomial = (
    Yu(f, h)
    * QClass.bar(index_labels={
        GENERATION_KIND: f,
        COLOR_FUND_KIND: c,
        SPINOR_KIND: alpha,
    })
    * QClass(index_labels={
        GENERATION_KIND: h,
        COLOR_FUND_KIND: c,
        SPINOR_KIND: alpha,
    })
    * Phi
)

lowered_yukawa = _lower_local_interaction_monomial(yukawa_monomial)

print("Declared monomial:")
print(yukawa_monomial)
print()
print("Lowered InteractionTerm:")
print(lowered_yukawa)
print()
print("Coupling:")
print(lowered_yukawa.coupling)
print()
print("Field occurrences:")
for occ in lowered_yukawa.fields:
    print(" ", occ)
print("Closed Dirac bilinears:", lowered_yukawa.closed_dirac_bilinears)


## 9. `InteractionTerm.to_vertex_kwargs()` is the boundary to the contraction engine

At this stage the project no longer talks about abstract field declarations only. It produces the exact parallel lists consumed by `vertex_factor(...)`: field species, external species, field roles, leg roles, `field_index_labels`, `field_index_types`, and derivative data.


In [20]:
legs = (
    QClass.leg(
        S("p1"),
        conjugated=True,
        spin=S("s1"),
        labels={
            GENERATION_KIND: S("f1"),
            COLOR_FUND_KIND: S("c1"),
            SPINOR_KIND: S("i1"),
        },
    ),
    QClass.leg(
        S("p2"),
        spin=S("s2"),
        labels={
            GENERATION_KIND: S("f2"),
            COLOR_FUND_KIND: S("c2"),
            SPINOR_KIND: S("i2"),
        },
    ),
    Phi.leg(S("p3")),
)

vertex_kwargs = lowered_yukawa.to_vertex_kwargs(legs)

print("Keys sent into vertex_factor(...):")
print(sorted(vertex_kwargs))

print()
print("Coupling sent to the engine:")
print(vertex_kwargs["coupling"])

print()
print("Field index types:")
for index_types in vertex_kwargs["field_index_types"]:
    print(" ", index_types)

print()
print("Field index labels:")
for labels in vertex_kwargs["field_index_labels"]:
    print(" ", labels)

print()
print("Leg index labels:")
for labels in vertex_kwargs["leg_index_labels"]:
    print(" ", labels)

print()
print("Closed Dirac bilinears:", vertex_kwargs["closed_dirac_bilinears"])

print()
print("Lower-level Feynman rule from the lowered InteractionTerm:")
print(lowered_yukawa.feynman_rule(QClass.bar, QClass, Phi, simplify=True, include_delta=True))


## 10. End-to-end gauge example

Now we let the standard `CovD` compiler use the same explicit index definitions. This shows how the declared field indices control the generated gauge-generator structure.


In [22]:
gS = S("gS")
g2 = S("g2")
mu = S("mu")

SU3_FUND = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=color_generator,
    name="fundamental",
)

SU2_FUND = GaugeRepresentation(
    index=WEAK_FUND_INDEX,
    generator_builder=weak_generator,
    name="doublet",
)

SU3C = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gS,
    gauge_boson="G",
    structure_constant=color_structure_constant,
    representations=(SU3_FUND,),
)

SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson="W",
    structure_constant=weak_structure_constant,
    representations=(SU2_FUND,),
)

qcd_model = Model(
    name="Explicit-index QCD demo",
    gauge_groups=(SU3C,),
    fields=(QClass, Gluon),
    lagrangian_decl=I * QClass.bar * Gamma(mu) * CovD(QClass, mu),
)

weak_model = Model(
    name="Explicit-index SU2L demo",
    gauge_groups=(SU2L,),
    fields=(LClass, WBoson),
    lagrangian_decl=I * LClass.bar * Gamma(mu) * CovD(LClass, mu),
)

u = QClass.class_members[0]

print("Generic qbar q G vertex:")
print(qcd_model.feynman_rule(QClass.bar, QClass, Gluon, simplify=True, include_delta=True))
print()
print("Flavor-expanded ubar u G vertex:")
print(
    qcd_model.feynman_rule(
        u.bar,
        u,
        Gluon,
        simplify=True,
        include_delta=True,
        flavor_expand=GENERATION_INDEX,
    )
)
print()
print("Generic Lbar L W vertex:")
print(weak_model.feynman_rule(LClass.bar, LClass, WBoson, simplify=True, include_delta=True))


## Summary

- The semantic index class used by the model layer is `IndexType`.
- The raw Spenso object underneath each declared index is a `Representation`.
- Concrete typed slots are `Slot` objects created from a `Representation`.
- Fields store tuples of `IndexType` declarations, not concrete labels.
- Concrete labels are attached later through `FieldOccurrence.labels` / `ExternalLeg.labels`.
- Lowering builds actual Spenso tensors with `TensorName.gamma()`, `TensorName.t()`, `TensorName.f()`, and representation metrics.
- `InteractionTerm.to_vertex_kwargs()` passes `field_index_types` and `field_index_labels` into the contraction engine.
- Repeated labels are contracted with `index.representation.g(...)`; open labels are remapped onto external-leg labels.
- Flavor is not special at the declaration layer: it is an `IndexType` with `is_flavor=True` and a finite `dimension`.
- One-index flavor parameters default to allowing the diagonal shorthand `y(f) * l.bar(f) * l(f)` via `Parameter.permits_label_summation()`.
